In [ ]:
import torch
from matplotlib import pyplot as plt
from tools.embs_tools import get_embs, aggregate_embeddings, accelerated_cosine_similarity, bb_weighted_average
import numpy as np
import itertools

In [ ]:
# all_embs = get_embs("/home/simon/car_outputs/")
# avg_fn = lambda embs: np.mean(embs, axis=0)
# all_embs_aggregated = {path: aggregate_embeddings(file_embs, aggregation_fn=avg_fn) for path, file_embs in all_embs.items()}

In [ ]:
all_embs = get_embs("/home/simon/new_outputs/")
all_embs_aggregated = {
    path: aggregate_embeddings(file_embs, aggregation_fn=bb_weighted_average)
    for path, file_embs in all_embs.items()
}

In [ ]:
embs = [torch.stack([torch.tensor(e) for _, e in embs]) for _, embs in all_embs_aggregated.items()]
embs = torch.stack(list(itertools.chain.from_iterable(embs)))[:100000]
similarity = accelerated_cosine_similarity(embs, embs, batch_size=524)
similarity = similarity.numpy()
similarity = similarity[np.triu_indices(similarity.shape[0], k=1)].flatten()

plt.hist(similarity, bins=100, range=(-1, 1), density=True)
plt.xlim([-0.5, 1.0])
plt.ylabel('Number of samples [mil]')
plt.xlabel('Cosine similarity score')
plt.title('Histogram of similarity scores')